### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 0

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 0
Seed set to 42


random seed set to 0
numpy seed set to 0
torch seed set to 0
lightning seed set to 0
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
    rating_threshold=4.0,
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.64,
    val_ratio=0.16,
    test_ratio=0.20,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.64 : 0.16 : 0.2):
train: 305206 (63.8%)
valid: 75578 (15.8%)
test: 97620 (20.41%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.547669
1    0.452331
Name: proportion, dtype: float64
valid label
0    0.606737
1    0.393263
Name: proportion, dtype: float64
test label
0    0.590914
1    0.409086
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 305206
  Num of positive interactions: 138054 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 276107], edge_label=[138054])
Edge Index: tensor([[    0,     0,     0,  ..., 10448, 10452, 10453],
        [ 2089,  2200,  2315,  ...,  1760,  1645,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:05<00:00, 371.56it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.340858,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.177372,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.457422,"[74.5, 34.0, 8.0, 5.0, 35.5, 45.5, 0.0, 37.0, ...",0.724296


#### Prepare Item Multihot Vec on Each Dimension

In [9]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1556,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
# train_triplet_with_dps_df.head(1)

valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
valid_triplet_with_dps_df = valid_triplet_df.merge(user_dps_df, on="userID", how="left")
valid_triplet_with_dps_df = valid_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="inner")
valid_triplet_with_dps_df.head(1)

Original data count (positive samples): 138054
Num of triplets: 138054(pos samples) * 5(negative sampled items) = 690270
Original data count (positive samples): 29722
Num of triplets: 29722(pos samples) * 5(negative sampled items) = 148610


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,4968,2844,"[334, 1644, 474, 755, 1206]",37,51,"[2, 9, 18, 0, 0, 0, 0, 0]","[1349, 1021, 1293, 1235, 1]",37,1,...,0.177372,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.457422,"[74.5, 34.0, 8.0, 5.0, 35.5, 45.5, 0.0, 37.0, ...",0.724296,4968,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."


### Prepare DataLoader

In [11]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
valid_dataset = TripletDataset(valid_triplet_with_dps_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))

# TODO: Custom sampler
# MIN_POS_ITEMS = 2
# user_to_pos_items, user_pos_to_indices = get_user_triplet_mapping(train_triplet_with_dps_df, MIN_POS_ITEMS)
# train_sampler = UserPosItemSampler(user_to_pos_items, user_pos_to_indices, batch_size=BATCH_SIZE, min_pos_items=MIN_POS_ITEMS, max_pos_items=20, buffer=0)
# train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, worker_init_fn=seed_worker, generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)


train data count: 690270
valid data count: 147820


### Configure Model (LightningModule)

In [12]:
from lightning_models.mtdp_ngcf_v2 import MTDPRecSRM

EMB_DIM = 64
LR = 1e-3
EPOCHS = 5
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

MT_WEIGHTS = {
    "bpr_loss": 1.0,
    "dps_loss": 0.5,
    "dpr_loss": 0.5,
    "dpm_loss": 0.5,
}

RESCALE_METHOD = None  # None, "log", "ema"

model = MTDPRecSRM(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD,
)


Seed set to 42


### Configure Trainer and Experiment

In [13]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "mtdp-ngcf-exp"
VERSION = "stat_test"
RUN_NAME = f"{RANDOM_SEED}"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_loss",
    monitor_mode="min",
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}",
)

In [14]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='gpu',  # or 'auto', 'gpu'
    devices=[1], # if gpu is available
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [15]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/mtdp-ngcf-exp exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                | Type              | Params | Mode 
-------------------------------------------------------------------
0  | ngcf_model          | NGCF              | 694 K  | train
1  | bpr_loss            | BPRLoss           | 0      | train
2  | reg_loss            | EmbLoss           | 0      | train
3  | dps_module          | DPSPredictor      | 1.0 K  | train
4  | dps_loss_fn         | DPSLoss           | 0      | train
5  | dpr_module          | DPRegularizer     | 262 K  | train
6  | dpr_loss_fn         | DPRLoss           | 0      | train
7  | dpm_module          | DPMatcher         | 0      | train
8  | dpm_loss_fn         | KLDivergenceLoss  | 0      | train
9  | loss_scaling_module | LogScal

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 3, global step 2700: 'val_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 3375: 'val_loss' was not in top 1
`Trainer.fit` stopped: `max_epochs=5` reached.


🏃 View run 0 at: http://140.112.106.216:3683/#/experiments/22/runs/2b44acb0459049deaef63ffb66d87974
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/22


### Inference

#### Prepare prediction pool for inference/testing

In [16]:
# NOTE: Prepare prediction pool to evaluate the model
seen_df = pd.concat([encoded_train_df, encoded_valid_df], ignore_index=True)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, seen_df, K=-1)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 8397, negative sampled to -1 items for each user
Num of interactions: 2064(users) * -1(items) = 16952432


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
16952427,2063,8392,0,"[2214, 1834, 1886, 2036, 1214]",37,1,"[6, 0, 0, 0, 0, 0, 0, 0]"
16952428,2063,8393,0,"[1070, 110, 1, 1, 1]",12,308,"[9, 0, 0, 0, 0, 0, 0, 0]"
16952429,2063,8394,0,"[1782, 1, 1, 1, 0]",36,365,"[4, 6, 0, 0, 0, 0, 0, 0]"
16952430,2063,8395,0,"[1, 1733, 935, 24, 1]",37,443,"[9, 17, 18, 0, 0, 0, 0, 0]"
16952431,2063,8396,0,"[1, 1, 0, 0, 0]",17,1,"[9, 0, 0, 0, 0, 0, 0, 0]"


In [17]:
from torch.utils.data import DataLoader
from common.datasets import UserItemPairDataset

test_dataset = UserItemPairDataset(prediction_pool_df)
print("test data count:", len(test_dataset))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


test data count: 16952432


In [18]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "mtdp-ngcf-v2-exp"
# best_model_checkpoint_path = ""
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = MTDPRecSRM.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.12755286693572998    │
│        test_ndcg20        │    0.1587136834859848     │
│        test_ndcg5         │    0.10018286854028702    │
│     test_precision10      │   0.036773256957530975    │
│     test_precision20      │   0.033527132123708725    │
│      test_precision5      │    0.04127907007932663    │
│       test_recall10       │     0.019526407122612     │
│       test_recall20       │    0.03678959235548973    │
│       test_recall5        │   0.011206962168216705    │
└───────────────────────────┴───────────────────────────┘

🏃 View run 0 at: http://140.112.106.216:3683/#/experiments/22/runs/2b44acb0459049deaef63ffb66d87974
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/22


[{'test_ndcg5': 0.10018286854028702,
  'test_ndcg10': 0.12755286693572998,
  'test_ndcg20': 0.1587136834859848,
  'test_precision5': 0.04127907007932663,
  'test_precision10': 0.036773256957530975,
  'test_precision20': 0.033527132123708725,
  'test_recall5': 0.011206962168216705,
  'test_recall10': 0.019526407122612,
  'test_recall20': 0.03678959235548973}]

In [19]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.100183,0.011207,0.041279,0.127553,0.019526,0.036773,0.158714,0.036790,0.033527
std,595.969798,0.239877,0.035715,0.103274,0.240728,0.049564,0.075176,0.233091,0.071092,0.056540
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1031.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1547.250000,0.000000,0.000000,0.000000,0.289065,0.010101,0.100000,0.301030,0.052632,0.050000
max,2063.000000,1.000000,0.500000,1.000000,1.000000,0.666667,0.600000,1.000000,0.750000,0.550000


In [20]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:04<00:00, 503.55it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.089715,0.963859,0.097507,0.803966,0.488762
std,595.969798,0.047038,0.053751,0.087182,0.097562,0.044698
min,0.000000,0.000000,0.339455,0.000000,0.313312,0.294328
25%,515.750000,0.055634,0.960421,0.019508,0.752760,0.461146
50%,1031.500000,0.088705,0.980974,0.083334,0.821521,0.491228
75%,1547.250000,0.121163,0.990432,0.150635,0.875349,0.520306
max,2063.000000,0.264863,1.000000,0.452302,0.978693,0.613322


In [21]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(eval_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.234213
std,20797.975208,0.056304
min,75.000000,0.073320
25%,17798.500000,0.195562
50%,35054.000000,0.233933
75%,53331.000000,0.272146
max,71534.000000,0.404497


In [22]:
# embedding_path = "embeddings/mtdp_ngcf/"
# os.makedirs(embedding_path, exist_ok=True)
# torch.save(model.user_emb.cpu(), f"{embedding_path}user_emb.pt")
# torch.save(model.item_emb.cpu(), f"{embedding_path}item_emb.pt")